# Predictor0916 — Colab 分步训练

本 Notebook 不调用 `training_main()`，而是直接使用 `Model`、`Dataset`、`MLP` 和 `Trainer`，按以下阶段逐步执行：

1. 数据与输出目录准备；
2. 下载并加载 Qwen LLM；
3. 处理 ForeLen 数据并加载训练/验证集；
4. 初始化 MLP 和 Trainer；
5. 执行训练；
6. 评估并保存实验结果。

默认模型为 `Qwen/Qwen2.5-0.5B-Instruct`，ForeLen subset 为 `qwen2.5-0.5b-longseq`。运行前请在 Colab 中选择 **运行时 → 更改运行时类型 → GPU**。

## 1. 安装依赖

In [ ]:
%pip install -q transformers datasets pandas numpy pyarrow accelerate sentencepiece huggingface_hub

## 2. 检查 Colab GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 CUDA GPU。请在 Colab 中选择：运行时 → 更改运行时类型 → GPU。"
    )

print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

## 3. 定位项目并导入底层组件

项目需要位于 `/content/Predictor0916`，或 `/content` 下某个目录的第一层。如果自动搜索失败，请手动修改 `PROJECT_DIR`。

In [ ]:
from pathlib import Path
import sys

project_candidates = [
    Path("/content/Predictor0916"),
    Path.cwd() / "Predictor0916",
    *Path("/content").glob("*/Predictor0916"),
]
PROJECT_DIR = next((path.resolve() for path in project_candidates if path.is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "找不到 Predictor0916。请先上传/克隆项目，或手动设置 PROJECT_DIR。"
    )

REPO_ROOT = PROJECT_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Predictor0916.src.dataset import Dataset
from Predictor0916.src.mlp import MLP
from Predictor0916.src.model import Model
from Predictor0916.src.trainer import Trainer
from Predictor0916.src.utils import (
    append_jsonl,
    compute_metrics,
    ensure_dir,
    save_json,
    set_seed,
)

print("Project directory:", PROJECT_DIR)

## 4. 可选：配置 Hugging Face 登录

可以在 Colab 左侧 **Secrets** 中创建 `HF_TOKEN`。Qwen 模型和 ForeLen 数据可公开访问时通常不要求登录，但配置 token 可以减少匿名请求限制。代码不会打印 token。

In [ ]:
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication configured.")
else:
    print("HF_TOKEN 未配置，将使用匿名访问。")

## 5. 实验参数

Colab T4 默认使用 `float16`。`MAX_SAMPLES=None` 表示处理完整 subset；首次调试时建议设为较小整数，例如 `100`。

In [ ]:
# LLM and ForeLen source
MODEL_ID_OR_PATH = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_SUBSET = "qwen2.5-0.5b-longseq"
SOURCE_SPLIT = "train"
LLM_DEVICE = "cuda:0"
MLP_DEVICE = "cuda:0"
TORCH_DTYPE = "float16"
LLM_BATCH_SIZE = 1
MAX_PROMPT_LENGTH = None
MAX_NEW_TOKENS = 8192
TRUST_REMOTE_CODE = True

# Local source data and processed feature data
LOCAL_SOURCE_PATH = PROJECT_DIR / "data" / "source" / DATASET_SUBSET / "train.parquet"
DATA_PATH = PROJECT_DIR / "data" / "qwen2.5-0.5b-longseq-generated.parquet"
HF_CACHE_DIR = PROJECT_DIR / "cache" / "huggingface"
MAX_SAMPLES = None  # Use a small integer for a quick Colab smoke test.
WRITER_BATCH_SIZE = 32
VALIDATION_RATIO = 0.2
NUM_WORKERS = 2
PIN_MEMORY = True

# MLP and training
OUTPUT_DIR = PROJECT_DIR / "outputs" / "qwen2.5-0.5b-longseq-colab"
LOAD_CHECKPOINT = None
NUM_BINS = 20
TARGET_QUANTILES = (0.01, 0.99)
LOSS_TYPE = "soft_label"
LAMBDA_VAL = 0.95
EPOCHS = 10
TRAIN_BATCH_SIZE = 256
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.0
PATIENCE = 3  # Set to None to disable early stopping.
SEED = 42

## 6. 准备本地数据文件

本 Notebook 不再从 Hugging Face 下载 ForeLen 数据。请先将 `qwen2.5-0.5b-longseq` 的原始 `train.parquet` 上传到 `LOCAL_SOURCE_PATH`。标准项目位置对应 `/content/Predictor0916/data/source/qwen2.5-0.5b-longseq/train.parquet`。已经生成过 `DATA_PATH` 时可以直接复用处理结果，不再要求原始文件。

In [ ]:
set_seed(SEED)
ensure_dir(LOCAL_SOURCE_PATH.parent)
ensure_dir(DATA_PATH.parent)
ensure_dir(HF_CACHE_DIR)
ensure_dir(OUTPUT_DIR)

if DATA_PATH.exists() and not DATA_PATH.is_file():
    raise IsADirectoryError(f"DATA_PATH 必须是文件路径：{DATA_PATH}")
if not DATA_PATH.is_file() and not LOCAL_SOURCE_PATH.is_file():
    raise FileNotFoundError(
        "未找到本地 ForeLen 数据文件。请上传到："
        f"{LOCAL_SOURCE_PATH}"
    )

print("Dataset subset:", DATASET_SUBSET)
print("Local source file:", LOCAL_SOURCE_PATH)
print("Local source exists:", LOCAL_SOURCE_PATH.is_file())
print("Processed data:", DATA_PATH)
print("Processed file exists:", DATA_PATH.is_file())
print("Output directory:", OUTPUT_DIR)

In [ ]:
# Optional Google Drive setup (run before changing DATA_PATH/OUTPUT_DIR).
# from google.colab import drive
# drive.mount("/content/drive")
# LOCAL_SOURCE_PATH = Path("/content/drive/MyDrive/Predictor0916/data/source/qwen2.5-0.5b-longseq/train.parquet")
# DATA_PATH = Path("/content/drive/MyDrive/Predictor0916/data/qwen2.5-0.5b-longseq-generated.parquet")
# OUTPUT_DIR = Path("/content/drive/MyDrive/Predictor0916/outputs/qwen2.5-0.5b-longseq-colab")

## 7. 下载并加载 Qwen 模型

`Model.load_model()` 会下载 tokenizer 和模型权重（缓存中不存在时），将模型移动到 GPU，并切换到 evaluation 模式。

In [ ]:
feature_model = Model(
    model_id_or_path=MODEL_ID_OR_PATH,
    batch_size=LLM_BATCH_SIZE,
    device=LLM_DEVICE,
    torch_dtype=TORCH_DTYPE,
    max_prompt_length=MAX_PROMPT_LENGTH,
    trust_remote_code=TRUST_REMOTE_CODE,
)
feature_model.load_model()
print(f"Loaded {MODEL_ID_OR_PATH} on {feature_model.device}")

## 8. 处理 ForeLen 数据

这里显式把已加载的 Qwen 模型和本地 `train.parquet` 注入 `Dataset`。如果处理后的目标 Parquet 已存在则复用；否则从 `LOCAL_SOURCE_PATH` 读取 prompt，逐条提取隐藏状态并生成回答长度，不会访问 Hugging Face 数据集服务。

In [ ]:
dataset = Dataset(
    model_id_or_path=MODEL_ID_OR_PATH,
    subset=DATASET_SUBSET,
    source_split=SOURCE_SPLIT,
    cache_dir=HF_CACHE_DIR,
    local_path=LOCAL_SOURCE_PATH,
    save_path=DATA_PATH,
    model=feature_model,
    model_batch_size=LLM_BATCH_SIZE,
    device=LLM_DEVICE,
    torch_dtype=TORCH_DTYPE,
    max_prompt_length=MAX_PROMPT_LENGTH,
    trust_remote_code=TRUST_REMOTE_CODE,
    max_new_tokens=MAX_NEW_TOKENS,
    generation_kwargs={"do_sample": False},
    seed=SEED,
)

if DATA_PATH.is_file():
    print("Reuse existing processed data:", DATA_PATH)
else:
    processed_path = dataset.process(
        max_samples=MAX_SAMPLES,
        writer_batch_size=WRITER_BATCH_SIZE,
        overwrite=False,
    )
    if processed_path != DATA_PATH:
        raise RuntimeError(f"Unexpected output path: {processed_path}")
    print("Processed data saved to:", processed_path)

## 9. 加载并划分训练/验证数据

In [ ]:
train_loader, validation_loader = dataset.load(
    validation_ratio=VALIDATION_RATIO,
    batch_size=TRAIN_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    seed=SEED,
)
train_features, train_targets = train_loader.dataset.tensors
validation_features, validation_targets = validation_loader.dataset.tensors

print("Train features:", tuple(train_features.shape))
print("Train targets:", tuple(train_targets.shape))
print("Validation features:", tuple(validation_features.shape))
print("Validation targets:", tuple(validation_targets.shape))

# Hidden states are already materialized on CPU, so release the LLM before
# placing the MLP and optimizer on the GPU.
feature_model.unload_model()
print("Released LLM weights from GPU.")

## 10. 初始化 MLP 和 Trainer

长度范围只使用训练集计算，避免验证集信息泄漏。MLP 参数由 PyTorch 在构造层时自动初始化；如果设置了 `LOAD_CHECKPOINT`，则随后严格加载已有 `self.layers` 参数。

In [ ]:
import numpy as np

low_quantile, high_quantile = TARGET_QUANTILES
if not 0.0 <= low_quantile < high_quantile <= 1.0:
    raise ValueError("TARGET_QUANTILES must satisfy 0 <= LOW < HIGH <= 1.")

range_low, range_high = np.quantile(
    train_targets.detach().cpu().numpy(),
    (low_quantile, high_quantile),
)
if range_high <= range_low:
    range_high = range_low + 1.0
target_range = (float(range_low), float(range_high))

head = MLP(
    input_dim=int(train_features.shape[1]),
    num_bins=NUM_BINS,
    target_range=target_range,
)
if LOAD_CHECKPOINT is not None:
    head.load(LOAD_CHECKPOINT)

trainer = Trainer(
    model=head,
    device=MLP_DEVICE,
    loss_type=LOSS_TYPE,
    lambda_val=LAMBDA_VAL,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=TRAIN_BATCH_SIZE,
    epochs=EPOCHS,
    patience=PATIENCE,
    seed=SEED,
)

print("Input dimension:", head.input_dim)
print("Target range:", target_range)
print("Trainable parameters:", sum(p.numel() for p in head.parameters() if p.requires_grad))

## 11. 训练

`Trainer.fit()` 会记录逐 epoch 指标，依据验证集 MAE 执行 early stopping，并在结束时恢复最佳参数。

In [ ]:
history = trainer.fit(
    train_features,
    train_targets,
    validation_features,
    validation_targets,
)
history

## 12. 验证并保存结果

这一阶段保存与 `test_training.py` 相同的五类实验产物：MLP 参数、逐样本预测、配置、指标/历史以及追加式 JSONL 记录。

In [ ]:
import pandas as pd

validation_predictions = trainer.predict(validation_features).numpy()
validation_targets_np = validation_targets.detach().cpu().numpy()
metrics = compute_metrics(validation_predictions, validation_targets_np)

checkpoint_path = OUTPUT_DIR / "checkpoints" / "best_layers.pt"
head.save(checkpoint_path)

predictions_path = OUTPUT_DIR / "validation_predictions.csv"
pd.DataFrame(
    {
        "sample_index": np.arange(len(validation_predictions)),
        "predicted_length": validation_predictions,
        "target_length": validation_targets_np,
        "absolute_error": np.abs(validation_predictions - validation_targets_np),
    }
).to_csv(predictions_path, index=False)

configuration = {
    "model_id_or_path": MODEL_ID_OR_PATH,
    "dataset_subset": DATASET_SUBSET,
    "source_split": SOURCE_SPLIT,
    "llm_batch_size": LLM_BATCH_SIZE,
    "llm_device": LLM_DEVICE,
    "torch_dtype": TORCH_DTYPE,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_samples": MAX_SAMPLES,
    "trust_remote_code": TRUST_REMOTE_CODE,
    "data_path": str(DATA_PATH),
    "validation_ratio": VALIDATION_RATIO,
    "input_dim": int(train_features.shape[1]),
    "num_bins": NUM_BINS,
    "target_quantiles": list(TARGET_QUANTILES),
    "target_range": list(target_range),
    "loss_type": LOSS_TYPE,
    "lambda_val": LAMBDA_VAL,
    "epochs": EPOCHS,
    "batch_size": TRAIN_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "device": MLP_DEVICE,
    "seed": SEED,
    "load_checkpoint": str(LOAD_CHECKPOINT) if LOAD_CHECKPOINT else None,
}
record = {
    "config": configuration,
    "split_sizes": {
        "train": len(train_targets),
        "validation": len(validation_targets),
    },
    "metrics": metrics,
    "history": history,
    "artifacts": {
        "checkpoint": str(checkpoint_path),
        "predictions": str(predictions_path),
    },
}

save_json(configuration, OUTPUT_DIR / "manifest.json")
save_json({"metrics": metrics, "history": history}, OUTPUT_DIR / "metrics.json")
append_jsonl([record], OUTPUT_DIR / "results.jsonl")

print("Saved checkpoint:", checkpoint_path)
print("Saved predictions:", predictions_path)

## 13. 查看指标和预测

In [ ]:
import json
from IPython.display import display

print("Split sizes:")
print(json.dumps(record["split_sizes"], indent=2, ensure_ascii=False))
print("Validation metrics:")
print(json.dumps(metrics, indent=2, ensure_ascii=False))
display(pd.read_csv(predictions_path).head(10))

## 14. 可选：打包并下载输出

输出未写入 Google Drive 时，可以取消下面代码的注释，将整个实验目录下载为 ZIP。

In [ ]:
# import shutil
# from google.colab import files
# archive_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
# files.download(archive_path)